[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/34_speculative_decoding.ipynb)

# 🔴 困难: 投机解码

实现投机解码的**接受/拒绝步骤**——一种加速 LLM 推理的技术。

### 函数签名
```python
def speculative_decode(target_probs, draft_probs, draft_tokens) -> list[int]:
    # target_probs: 来自目标（大）模型的 (K, V)
    # draft_probs: 来自草稿（小）模型的 (K, V)
    # draft_tokens: 草稿模型采样的 (K,) tokens
    # 返回: 接受的 token 列表 (1 到 K)
```

### 算法
对于每个位置 i = 0, ..., K-1:
1. `ratio = target_probs[i, token_i] / draft_probs[i, token_i]`
2. 以概率 `min(1, ratio)` 接受
3. 如果拒绝：从 `normalize(max(0, target - draft))` 采样，追加并停止

In [ ]:
# 在 Colab 中安装 torch-judge（在 JupyterLab/Docker 中无操作）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✏️ 在此实现你的代码

def speculative_decode(target_probs, draft_probs, draft_tokens):
    pass  # 接受/拒绝循环

## 投机解码（Speculative Decoding）原理解析

投机解码是一种加速大语言模型（LLM）推理的创新技术，让我详细讲解其核心原理。

### 1. 核心思想

#### 问题背景
- **大模型**：生成质量高但推理速度慢（每次生成一个token都需要完整前向传播）
- **小模型**：推理速度快但生成质量相对较低

#### 基本思路
用<strong>小模型（草稿模型）</strong>快速生成多个候选token，再用<strong>大模型（目标模型）</strong>一次性验证这些token的正确性，从而在保证质量的同时加速生成。

### 2. 工作流程

```
输入 → 草稿模型生成K个候选token → 目标模型验证 → 接受/拒绝 → 输出
```

#### 详细步骤：

```
步骤1: 草稿生成阶段
- 使用小模型快速生成K个候选token: [t₁, t₂, t₃, ..., t_K]
- 记录每个位置的草稿概率分布: draft_probs[i]

步骤2: 目标验证阶段  
- 将草稿token序列输入大模型
- 一次前向传播得到所有位置的目标概率分布: target_probs[i]

步骤3: 接受/拒绝决策（关键步骤）
- 对每个位置i，计算接受概率: αᵢ = min(1, target_probs[i][tᵢ] / draft_probs[i][tᵢ])
- 以概率αᵢ接受草稿token tᵢ
- 一旦拒绝，立即停止并从修正分布采样

步骤4: 修正采样（拒绝时）
- 从 residual = normalize(max(0, target_probs[i] - draft_probs[i])) 采样
- 这个修正保证了输出分布的正确性
```

### 3. 数学原理

#### 为什么接受概率这样计算？

关键在于**重要性采样**的思想：

```
接受概率 α = min(1, p_target(t) / p_draft(t))
```

- 如果草稿模型对token t的概率**低于**目标模型，说明草稿模型"低估"了这个token，我们应该更可能接受它
- 如果草稿模型对token t的概率**高于**目标模型，说明草稿模型"高估"了这个token，我们需要以相应概率拒绝

#### 修正分布的正确性

拒绝时从 `max(0, p_target - p_draft)` 采样，保证了：

```
p_target = p_draft * α + (1-α) * residual_normalized
```

这确保了最终采样分布**精确等于**目标模型的分布！

### 4. 性能分析

#### 加速原理
- **传统方法**：生成K个token需要K次大模型前向传播
- **投机解码**：生成K个token只需要1次大模型前向传播 + K次小模型前向传播

#### 关键指标
```
期望接受长度 = Σ P(接受前i个token)
加速比 ≈ 期望接受长度 / (1 + 草稿模型开销比)
```

#### 最优草稿长度K
- K太小：验证次数多，加速有限
- K太大：拒绝概率增加，浪费计算
- 通常 K= 3-5 是最优选择

### 5. 实际应用注意事项

#### 草稿模型选择
- **太小**：生成质量差，接受率低
- **太大**：推理开销大，加速效果差
- **最佳**：与目标模型架构相似，但参数量小10-100倍

#### 实现优化
1. **批处理**：同时处理多个验证位置
2. **KV缓存**：复用草稿模型的KV缓存
3. **动态K**：根据接受率自适应调整草稿长度

#### 局限性
- 需要额外内存存储两个模型
- 草稿模型质量影响最终效率
- 对某些任务（如数学推理）可能效果不佳

### 6. 总结

投机解码通过"快速草稿+慢速验证"的范式，在保证输出质量的前提下显著加速LLM推理。这个技术的精妙之处在于：

1. **理论保证**：通过精心设计的接受/拒绝机制，确保输出分布精确等于目标模型
2. **实际效果**：在实践中可实现2-3倍的推理加速
3. **通用性**：不改变模型结构，可应用于各种LLM

这正是为什么投机解码已成为现代LLM推理系统（如vLLM、TensorRT-LLM等）的标准优化技术之一。

In [ ]:
def speculative_decode_vectorized(target_probs, draft_probs, draft_tokens) -> list[int]:
    '''
    投机解码矩阵
    预测目标（大）模型，并使用预测结果来决定草稿（小）模型是否接受预测结果。
    期望接受长度 = Σ P(接受前i个token)
    加速比 ≈ 期望接受长度 / (1 + 草稿模型开销比)

    args:
        - target_probs: 来自目标（大）模型输出的 K 个 token 的概率分布(非对数概率)  (K, V)
        - draft_probs: 来自草稿（小）模型输出的 K 个 token 的对数概率分布(非对数概率)  (K, V)
        - draft_tokens: 草稿模型采样的 K 个 token 的索引 (K,)

    returns:
        - 接受的 token 列表 (1 到 K)
    '''

    K = draft_tokens.shape[0] # 每次预测 K 个 token, 通过 outpt_shape = (B, S, V) 每次根据最后一个 S[-1] 的 V 获取 token，连续推理 K step.
    device = target_probs.device # 草稿模型的硬件设备
    
    # 根据草稿模型输出的每个 token 索引收集目标模型和草稿模型对应输出的的概率
    draft_token_probs = draft_probs[torch.arange(K), draft_tokens] # (K,)
    target_token_probs = target_probs[torch.arange(K), draft_tokens] # (K,)
    
    # 计算接受概率（处理除零，防止除数为零）
    safe_mask = draft_token_probs > 0
    ratios = torch.zeros_like(draft_token_probs)
    ratios[safe_mask] = target_token_probs[safe_mask] / draft_token_probs[safe_mask] # 计算每个 token 的概率比，即目标模型的概率 / 草稿模型的概率
    accept_probs = torch.min(torch.ones_like(ratios), ratios) # 获取接受概率,超过 1 的取 1
    
    # 生成随机数决定每个位置是否接受
    random_values = torch.rand(K, device=device)
    accept_mask = random_values < accept_probs # 获得接受的 token 位置, (K, ) 

    # 找到第一个拒绝的位置，选择这个位置的 token, 以让草稿模型根据残差分布采样的新的 token 继续 step.如果掠过 first_reject_idx,下一个 step 继续计算出相同的概率分布,这样无法跨越这个拒绝的位置.
    if torch.all(accept_mask):
        # 全部接受
        return draft_tokens.tolist()
    else:
        # 找到第一个拒绝的位置
        first_reject_idx = torch.where(~accept_mask)[0][0].item() # torch.where 返回一个元组，包含所有满足条件的索引, 取第一行的第 0 个索引, .item() 转换为标量
        
        # 接受所有在拒绝位置之前的token
        accepted_tokens = draft_tokens[:first_reject_idx].tolist()
        
        # 从目标模型大于草稿模型的所有 token 中重新采样一个 token 来替换它。从残差分布中采样一个token, 这个修正分布被称为残差分布（residual distribution）
        if first_reject_idx < K:
            residual = torch.clamp(target_probs[first_reject_idx] - draft_probs[first_reject_idx], min=0) # torch.clamp 限制最小值, 在第一个不接受的位置处只保留目标模型概率大于草稿模型概率的部分
            residual_sum = residual.sum()
            
            if residual_sum > 0:
                residual_normalized = residual / residual_sum # 将词典概率分布归一化
                sampled_token = torch.multinomial(residual_normalized, 1).item() # 根据残差概率随机采样一个 token
            else:
                sampled_token = torch.randint(0, len(target_probs[first_reject_idx]), (1,)).item() # 如果目标模型的词典中的每个 token 的概率都小于草稿模型，则随机取一个 token
            
            accepted_tokens.append(sampled_token)
        
        return accepted_tokens

In [ ]:
# 错误做法
accepted_tokens = draft_tokens[:first_reject_idx]  # 不接受任何token
# 然后继续从下一个位置开始

# 错误做法
sampled_token = torch.multinomial(target_probs[first_reject_idx], 1) # 这看起来合理，但实际上破坏了概率分布的数学一致性。

In [ ]:
# 🧪 调试
torch.manual_seed(0)
probs = torch.softmax(torch.randn(4, 10), dim=-1)
tokens = torch.tensor([2, 5, 1, 8])
print('完美草稿:', speculative_decode(probs, probs, tokens))
target = torch.softmax(torch.randn(4, 10), dim=-1)
draft = torch.softmax(torch.randn(4, 10), dim=-1)
print('随机草稿:', speculative_decode(target, draft, tokens))

In [ ]:
# ✅ 提交
from torch_judge import check
check('speculative_decoding')